# Receiver Windows and Actionable Chances

**Receiver window:** receiver reaches first-down line, has at least 3.5 yards separation at decision frame, retains at least 2 projected yards after 0.5 seconds, and holds conditions for about 0.3 seconds.

**Actionable chance:** receiver window on a play with no PFF hurry, hit, or sack.

**Actionability rate:** actionable chances divided by receiver-window plays. It measures how often protection and play context preserve a created window. It is not receiver skill and not a pure offensive-line grade.

Current validation: 32.2% first-down success without an actionable chance versus 52.5% with one.

In [62]:
from pathlib import Path
import pandas as pd
from IPython.display import HTML, IFrame, display

ROOT = Path.cwd()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent
OUTPUT = ROOT / 'chance_stat' / 'output'

routes = pd.read_csv(OUTPUT / 'route_chances.csv')
receivers = pd.read_csv(OUTPUT / 'receiver_scores.csv')
qb_games = pd.read_csv(OUTPUT / 'qb_game_summary.csv')
team_games = pd.read_csv(OUTPUT / 'team_game_summary.csv')
defense = pd.read_csv(OUTPUT / 'defense_summary.csv')

## Dataset summary

In [63]:
dataset_summary = pd.Series({
    'Games': routes['gameId'].nunique(),
    'Scored plays': routes[['gameId', 'playId']].drop_duplicates().shape[0],
    'Routes': len(routes),
    'Receivers': routes['nflId'].nunique(),
    'Receiver-window plays': routes.groupby(['gameId', 'playId'])['team_play_window'].max().sum(),
    'Actionable-chance plays': routes.groupby(['gameId', 'playId'])['actionable_chance'].max().sum(),
    'QB-game observations': len(qb_games),
    'Quarterbacks': qb_games['qb_key'].nunique(),
    'Average yards per dropback': round(
        (qb_games['yards_per_dropback'] * qb_games['dropbacks']).sum() / qb_games['dropbacks'].sum(), 2
    ),
}, name='Value')
display(dataset_summary.to_frame())

,Value
Games,122.0
Scored plays,8533.0
Routes,39398.0
Receivers,476.0
Receiver-window plays,2482.0
Actionable-chance plays,1344.0
QB-game observations,280.0
Quarterbacks,60.0
Average yards per dropback,6.5


## Play-level validation

In [64]:
play_level = routes.groupby(['gameId', 'playId'], as_index=False).agg(
    receiver_window=('team_play_window', 'max'),
    actionable_chance=('actionable_chance', 'max'),
    pressured=('pressured', 'max'),
    play_result=('playResult', 'first'),
    yards_to_go=('yardsToGo', 'first'),
)
play_level['first_down'] = (play_level['play_result'] >= play_level['yards_to_go']).astype(int)
play_level['state'] = 'No receiver window'
play_level.loc[play_level['receiver_window'].eq(1), 'state'] = 'Window blocked by pressure'
play_level.loc[play_level['actionable_chance'].eq(1), 'state'] = 'Actionable chance'
validation = play_level.groupby('state').agg(
    plays=('playId', 'size'),
    first_down_rate=('first_down', 'mean'),
    yards_per_dropback=('play_result', 'mean'),
)
validation['first_down_rate'] = (100 * validation['first_down_rate']).round(1)
validation['yards_per_dropback'] = validation['yards_per_dropback'].round(2)
display(validation)
print(f"Overall actionability: {100 * play_level['actionable_chance'].sum() / play_level['receiver_window'].sum():.1f}%")

,plays,first_down_rate,yards_per_dropback
state,,,
Actionable chance,1344,52.5,9.68
No receiver window,6051,33.0,6.40
Window blocked by pressure,1138,28.2,3.76


Overall actionability: 54.1%


## Original V1 validation baseline

Before tightening chance definition and adding pressure adjustment, original broad window produced:

| Original play state | First-down success | Average yards/dropback |
|---|---:|---:|
| No receiver chance | **31.0%** | **6.37** |
| At least one receiver chance | **41.8%** | **6.84** |

Original uplift: **10.7 percentage points**. Keep this as baseline comparison, not current formula result.

## Receiver leaderboard

In [65]:
receiver_leaderboard = receivers.loc[receivers['routes'] >= 100, [
    'displayName', 'officialPosition', 'routes', 'receiver_windows',
    'credited_windows', 'chance_created_per_100_routes'
]].sort_values('chance_created_per_100_routes', ascending=False).head(20).copy()
receiver_leaderboard[['credited_windows', 'chance_created_per_100_routes']] = receiver_leaderboard[['credited_windows', 'chance_created_per_100_routes']].round(1)
display(receiver_leaderboard)

,displayName,officialPosition,routes,receiver_windows,credited_windows,chance_created_per_100_routes
16,Marquise Brown,WR,234,53,37.1,15.8
19,Quez Watkins,WR,167,32,25.4,15.2
25,DK Metcalf,WR,204,40,27.2,13.4
26,Michael Pittman,WR,249,42,32.6,13.1
27,Sterling Shepard,WR,144,23,18.8,13.1
28,Brandin Cooks,WR,236,40,30.8,13.1
29,Justin Jefferson,WR,240,40,30.8,12.8
31,Marvin Jones,WR,240,43,30.2,12.6
34,Tyler Lockett,WR,216,41,26.8,12.4
35,Kendall Hinton,WR,101,16,12.5,12.4


## Offensive-line actionability proxy

`Actionability rate = actionable chances / receiver-window plays`. Higher rate means offense preserved more created windows without a PFF hurry, hit, or sack. This is a protection-unit proxy, not a pure offensive-line grade: QB timing, blitz pickup, running backs, tight ends, and play design also affect pressure.

In [66]:
offensive_line_proxy = team_games.groupby('possessionTeam', as_index=False).agg(
    games=('gameId', 'nunique'),
    dropbacks=('dropbacks', 'sum'),
    receiver_windows=('team_windows', 'sum'),
    actionable_chances=('actionable_chances', 'sum'),
    pressured_dropbacks=('pressured_dropbacks', 'sum'),
).rename(columns={'possessionTeam': 'Team'})
offensive_line_proxy['windows_blocked_by_pressure'] = offensive_line_proxy['receiver_windows'] - offensive_line_proxy['actionable_chances']
offensive_line_proxy['actionability_rate'] = 100 * offensive_line_proxy['actionable_chances'] / offensive_line_proxy['receiver_windows']
offensive_line_proxy['pressure_rate'] = 100 * offensive_line_proxy['pressured_dropbacks'] / offensive_line_proxy['dropbacks']
offensive_line_proxy[['actionability_rate', 'pressure_rate']] = offensive_line_proxy[['actionability_rate', 'pressure_rate']].round(1)
offensive_line_proxy = offensive_line_proxy.sort_values('actionability_rate', ascending=False)
display(offensive_line_proxy.reset_index(drop=True))

,Team,games,dropbacks,receiver_windows,actionable_chances,pressured_dropbacks,windows_blocked_by_pressure,actionability_rate,pressure_rate
0,ATL,7,288,68,46,108,22,67.6,37.5
1,TB,8,309,77,52,81,25,67.5,26.2
2,LA,8,249,81,53,70,28,65.4,28.1
3,PIT,7,238,37,24,65,13,64.9,27.3
4,NE,8,284,65,42,94,23,64.6,33.1
5,NYG,8,299,79,48,112,31,60.8,37.5
6,WAS,8,298,93,56,105,37,60.2,35.2
7,DAL,7,261,81,48,84,33,59.3,32.2
8,ARI,8,233,79,46,86,33,58.2,36.9
9,MIN,7,254,69,40,95,29,58.0,37.4


## QB actionability stability

Minimum 100 dropbacks removes backup-QB samples.

In [67]:
qb_actionability = qb_games.groupby(['qb_key', 'quarterback', 'possessionTeam'], as_index=False).agg(
    games=('gameId', 'size'),
    dropbacks=('dropbacks', 'sum'),
    receiver_windows=('receiver_windows', 'sum'),
    actionable_chances=('actionable_chances', 'sum'),
)
qb_actionability = qb_actionability[qb_actionability['dropbacks'] >= 100].copy()
qb_actionability['window_rate'] = 100 * qb_actionability['receiver_windows'] / qb_actionability['dropbacks']
qb_actionability['actionability_rate'] = 100 * qb_actionability['actionable_chances'] / qb_actionability['receiver_windows']
qb_actionability[['window_rate', 'actionability_rate']] = qb_actionability[['window_rate', 'actionability_rate']].round(1)
display(qb_actionability.sort_values('actionability_rate', ascending=False))
print(f"Mean actionability: {qb_actionability['actionability_rate'].mean():.1f}%")
print(f"Standard deviation: {qb_actionability['actionability_rate'].std():.1f} points")
print(f"Window-rate/actionability correlation: {qb_actionability['window_rate'].corr(qb_actionability['actionability_rate']):.2f}")

,qb_key,quarterback,possessionTeam,games,dropbacks,receiver_windows,actionable_chances,window_rate,actionability_rate
4,33084,Matt Ryan,ATL,7,286,66,46,23.1,69.7
0,25511,Tom Brady,TB,8,306,77,52,25.2,67.5
7,34452,Matthew Stafford,LA,8,245,81,53,33.1,65.4
1,28963,Ben Roethlisberger,PIT,7,238,37,24,15.5,64.9
58,53444,Mac Jones,NE,8,281,65,42,23.1,64.6
13,38605,Russell Wilson,SEA,5,130,48,30,36.9,62.5
44,47789,Daniel Jones,NYG,8,277,77,48,27.8,62.3
24,42831,Taylor Heinicke,WAS,8,290,92,56,31.7,60.9
43,47784,Kyler Murray,ARI,8,233,79,46,33.9,58.2
28,43424,Dak Prescott,DAL,6,215,74,43,34.4,58.1


Mean actionability: 54.8%
Standard deviation: 6.7 points
Window-rate/actionability correlation: -0.31


## QB-game summary

In [68]:
qb_summary_source = qb_games.copy()
qb_summary_source['total_yards_estimate'] = qb_summary_source['yards_per_dropback'] * qb_summary_source['dropbacks']
qb_summary_source['first_downs_estimate'] = qb_summary_source['first_down_rate'] * qb_summary_source['dropbacks'] / 100
qb_summary = qb_summary_source.groupby(['quarterback', 'possessionTeam'], as_index=False).agg(
    games=('gameId', 'size'),
    dropbacks=('dropbacks', 'sum'),
    receiver_windows=('receiver_windows', 'sum'),
    actionable_chances=('actionable_chances', 'sum'),
    first_downs=('first_downs_estimate', 'sum'),
    total_yards=('total_yards_estimate', 'sum'),
)
qb_summary['window_rate'] = 100 * qb_summary['receiver_windows'] / qb_summary['dropbacks']
qb_summary['actionable_chance_rate'] = 100 * qb_summary['actionable_chances'] / qb_summary['dropbacks']
qb_summary['actionability_rate'] = 100 * qb_summary['actionable_chances'] / qb_summary['receiver_windows']
qb_summary['first_down_rate'] = 100 * qb_summary['first_downs'] / qb_summary['dropbacks']
qb_summary['yards_per_dropback'] = qb_summary['total_yards'] / qb_summary['dropbacks']
rate_columns = ['window_rate', 'actionable_chance_rate', 'actionability_rate', 'first_down_rate', 'yards_per_dropback']
qb_summary[rate_columns] = qb_summary[rate_columns].round(1)
display(qb_summary.sort_values('dropbacks', ascending=False).head(20))

,quarterback,possessionTeam,games,dropbacks,receiver_windows,actionable_chances,first_downs,total_yards,window_rate,actionable_chance_rate,actionability_rate,first_down_rate,yards_per_dropback
22,Jared Goff,DET,8,325,87,44,104.0,1725.0,26.8,13.5,50.6,32.0,5.3
43,Patrick Mahomes,KC,8,322,104,54,127.0,2211.0,32.3,16.8,51.9,39.4,6.9
52,Tom Brady,TB,8,306,77,52,121.0,2314.0,25.2,17.0,67.5,39.5,7.6
49,Taylor Heinicke,WAS,8,290,92,56,96.0,1825.0,31.7,19.3,60.9,33.1,6.3
37,Matt Ryan,ATL,7,286,66,46,96.0,1709.0,23.1,16.1,69.7,33.6,6.0
36,Mac Jones,NE,8,281,65,42,87.0,1729.0,23.1,14.9,64.6,31.0,6.2
12,Daniel Jones,NYG,8,277,77,48,89.0,1777.0,27.8,17.3,62.3,32.1,6.4
47,Ryan Tannehill,TEN,8,277,65,36,96.0,1693.0,23.5,13.0,55.4,34.7,6.1
48,Sam Darnold,CAR,8,276,78,39,89.0,1686.0,28.3,14.1,50.0,32.2,6.1
32,Justin Herbert,LAC,7,274,66,34,98.0,1823.0,24.1,12.4,51.5,35.8,6.7


## QB interactive dashboard

In [69]:
display(IFrame(src='output/qb_dashboard.html', width='100%', height=900))

## Defensive evaluation validation

Receiver-window rate isolates coverage. Actionable-chance rate also credits pass rush. Positive correlation with conceded outcomes is expected if metric captures dangerous offensive opportunities.

In [70]:
defensive_validation = pd.DataFrame({
    'Metric': ['Receiver-window rate allowed', 'Actionable-chance rate allowed'],
    'First-down correlation': [
        defense['window_rate_allowed'].corr(defense['first_down_rate_conceded']),
        defense['actionable_chance_rate_allowed'].corr(defense['first_down_rate_conceded']),
    ],
    'TD correlation': [
        defense['window_rate_allowed'].corr(defense['touchdown_rate_conceded']),
        defense['actionable_chance_rate_allowed'].corr(defense['touchdown_rate_conceded']),
    ],
    'Yards/dropback correlation': [
        defense['window_rate_allowed'].corr(defense['yards_per_dropback_allowed']),
        defense['actionable_chance_rate_allowed'].corr(defense['yards_per_dropback_allowed']),
    ],
})
defensive_validation.iloc[:, 1:] = defensive_validation.iloc[:, 1:].round(3)
display(defensive_validation)

,Metric,First-down correlation,TD correlation,Yards/dropback correlation
0,Receiver-window rate allowed,0.164,0.032,-0.036
1,Actionable-chance rate allowed,0.267,0.140,0.081


## Defense table and plots

Each row and point represents one defense. Plots use actionable chances so pass-rush success receives credit.

In [71]:
defense_view = defense[[
    'defensiveTeam', 'window_rate_allowed', 'actionable_chance_rate_allowed',
    'actionability_rate_allowed', 'first_down_rate_conceded', 'touchdown_rate_conceded'
]].rename(columns={
    'defensiveTeam': 'Team',
    'window_rate_allowed': 'Opponent receiver windows per 100 dropbacks',
    'actionable_chance_rate_allowed': 'Opponent actionable chances per 100 dropbacks',
    'actionability_rate_allowed': 'Actionable percentage of windows',
    'first_down_rate_conceded': 'First downs conceded per 100 dropbacks',
    'touchdown_rate_conceded': 'TDs conceded per 100 dropbacks',
}).sort_values('Opponent actionable chances per 100 dropbacks')
defense_view.iloc[:, 1:] = defense_view.iloc[:, 1:].round(2)
display(defense_view.reset_index(drop=True))

,Team,Opponent receiver windows per 100 dropbacks,Opponent actionable chances per 100 dropbacks,Actionable percentage of windows,First downs conceded per 100 dropbacks,TDs conceded per 100 dropbacks
0,CHI,21.16,8.71,41.18,33.61,6.22
1,MIN,23.32,11.46,49.15,32.41,3.16
2,DAL,21.07,11.49,54.55,34.10,6.90
3,BAL,22.48,11.63,51.72,35.27,4.65
4,WAS,24.03,11.69,48.65,41.23,6.82
5,LAC,25.11,11.89,47.37,31.28,2.64
6,NYJ,21.05,12.15,57.69,34.41,3.24
7,DEN,21.89,12.45,56.90,33.58,3.40
8,JAX,28.25,12.56,44.44,36.77,4.48
9,NE,25.89,12.77,49.32,31.56,3.90


In [72]:
def defense_scatter_svg(data, y_column, title):
    x_column = 'Opponent actionable chances per 100 dropbacks'
    width, height, left, right, top, bottom = 720, 430, 72, 24, 42, 62
    xs, ys = data[x_column].astype(float), data[y_column].astype(float)
    x_pad = max((xs.max() - xs.min()) * 0.12, 1)
    y_pad = max((ys.max() - ys.min()) * 0.15, 0.25)
    x_min, x_max = xs.min() - x_pad, xs.max() + x_pad
    y_min, y_max = ys.min() - y_pad, ys.max() + y_pad
    sx = lambda value: left + (value - x_min) / (x_max - x_min) * (width - left - right)
    sy = lambda value: height - bottom - (value - y_min) / (y_max - y_min) * (height - top - bottom)
    parts = [f'<div style="max-width:{width}px"><h3>{title}</h3><svg viewBox="0 0 {width} {height}" style="width:100%">']
    for index in range(5):
        x_tick = x_min + index * (x_max - x_min) / 4
        y_tick = y_min + index * (y_max - y_min) / 4
        parts += [f'<line x1="{sx(x_tick):.1f}" y1="{top}" x2="{sx(x_tick):.1f}" y2="{height-bottom}" stroke="#ddd"/>', f'<text x="{sx(x_tick):.1f}" y="{height-bottom+20}" text-anchor="middle" font-size="11">{x_tick:.1f}</text>', f'<line x1="{left}" y1="{sy(y_tick):.1f}" x2="{width-right}" y2="{sy(y_tick):.1f}" stroke="#ddd"/>', f'<text x="{left-8}" y="{sy(y_tick)+4:.1f}" text-anchor="end" font-size="11">{y_tick:.1f}</text>']
    for _, row in data.iterrows():
        x, y, team = float(row[x_column]), float(row[y_column]), row['Team']
        parts += [f'<circle cx="{sx(x):.1f}" cy="{sy(y):.1f}" r="5" fill="#176b87"><title>{team}: {x:.1f}, {y:.1f}</title></circle>', f'<text x="{sx(x)+7:.1f}" y="{sy(y)-6:.1f}" font-size="10">{team}</text>']
    parts += [f'<text x="{width/2:.1f}" y="{height-10}" text-anchor="middle" font-size="12">{x_column}</text>', f'<text transform="translate(15 {height/2:.1f}) rotate(-90)" text-anchor="middle" font-size="12">{y_column}</text>', '</svg></div>']
    return ''.join(parts)

display(HTML(defense_scatter_svg(defense_view, 'First downs conceded per 100 dropbacks', 'Actionable chances vs first downs conceded')))
display(HTML(defense_scatter_svg(defense_view, 'TDs conceded per 100 dropbacks', 'Actionable chances vs TDs conceded')))
print('Correlations:')
print('First downs:', round(defense['actionable_chance_rate_allowed'].corr(defense['first_down_rate_conceded']), 3))
print('TDs:', round(defense['actionable_chance_rate_allowed'].corr(defense['touchdown_rate_conceded']), 3))

Correlations:
First downs: 0.267
TDs: 0.14


## Interpretation limits

- Actionability is not a pure offensive-line grade as depends also on team tactics and opponent tactics.
- Eight games per primary QB makes game-level correlations noisy.
- Touchdowns cover tracked dropbacks only.
- Correlation supports relevance but does not prove causal attribution.